# 01 · Create training data with SDG Hub

**What you will do**
1. Set up the project volume and download the base model.
2. Look at the two kinds of synthetic data used in this example.
3. Write down the rules as code: a **report checker** and a **burst-line checker**.
4. Run an **SDG Hub flow**: a teacher model writes burst lines, and the checker keeps only the ones that follow every rule.

**Time:** about 15 minutes (plus the teacher's generation time). **GPU:** none in this notebook.

| | Dev environment | On OpenShift AI |
|---|---|---|
| Data pipeline | One-off scripts | SDG Hub flows: YAML files of reusable blocks, built-in flows to start from |
| Teacher model | Outside API, data leaves your environment | Any OpenAI-compatible model, ideally served in this project |
| Packages | Pinned by hand | SDG Hub comes from Red Hat's vetted package index |

> **The scenario is fictional.** Blue Defence runs Sentinel-7 uncrewed patrol vessels. Operators send terse radio
> notes; the ops room needs exact JSON reports (BD-SCR), and the shore needs one short satellite message per report
> (the *burst line*). All data is synthetic.

## 1 · Settings

In [ ]:
# Settings used by every notebook. Change them here if your project differs.
import os, json, time
from pathlib import Path

NAMESPACE  = open("/var/run/secrets/kubernetes.io/serviceaccount/namespace").read().strip()   # this project
WS         = Path("/opt/app-root/src/workspace")   # the project volume, as mounted in this workbench
PVC        = "workspace"                           # its name; training and serving pods mount it at /workspace
REPO       = Path.cwd().parent                     # this repository (data/, sdg_flows/)
BASE_MODEL = "ibm-granite/granite-4.1-3b"
EXPERIMENT = "sentinel7"                           # MLflow experiment
NODE_SELECTOR = {}   # only with ReadWriteOnce storage: keep every pod that mounts the volume on one node
TRAINING_IMAGE = ("registry.redhat.io/rhoai/odh-th-torch-cuda-py312-rhel9@sha256:"   # OpenShift AI 3.5 Training Hub image
                  "884aeeb039f9592252fe0c518acf31803a208757d4fa6500fe79884a022ea52d")

os.environ.setdefault("MLFLOW_EXPERIMENT_NAME", EXPERIMENT)
print("project:", NAMESPACE, "| volume:", WS, "| MLflow:", os.environ.get("MLFLOW_TRACKING_URI", "not enabled"))

## 2 · The project volume and the base model

The volume holds everything the later notebooks share: the base model, data, checkpoints and exports.
Create it once with `deploy/00-project-setup.yaml`, and attach it to this workbench at `/opt/app-root/src/workspace`
(Workbench settings > Cluster storage).

In [ ]:
for sub in ["models", "data", "checkpoints", "results", "export", "code"]:
    (WS / sub).mkdir(parents=True, exist_ok=True)

# Copy the synthetic data from this repository onto the volume, where training pods can read it
import shutil
for d in ["edgecopilot", "burst"]:
    shutil.copytree(REPO / "data" / d, WS / "data" / d, dirs_exist_ok=True)
print(sorted(str(p.relative_to(WS)) for p in (WS / "data").rglob("*.jsonl")))

In [ ]:
# Download Granite 4.1-3B onto the volume (about 7 GB, a few minutes). Skip if it is already there.
from huggingface_hub import snapshot_download
target = WS / "models" / "granite-4.1-3b"
if not (target / "config.json").exists():
    snapshot_download(BASE_MODEL, local_dir=target,
                      allow_patterns=["*.json", "*.safetensors", "*.txt", "*.model", "*.jinja"])
print(sorted(p.name for p in target.iterdir()))

## 3 · Two kinds of synthetic data

| Use case | Data | How it was made |
|---|---|---|
| **Edge Copilot** (SFT): radio note → exact JSON report | `data/edgecopilot/` · 2,400 train, 150 val, 300 test | **Programmatic.** A script draws a correct report, then writes it as a messy note. The correct answer is known exactly. `train-v2` adds 600 radio-style notes. |
| **Burst line** (GRPO): report → one satellite message | `data/burst/` · 400 train, 100 val, 200 test | **SDG Hub.** A teacher model writes the messages; a rule checker keeps only correct ones. |

Programmatic data is ideal when you can generate the answer. When many answers are valid, a teacher model plus a
checker is the practical route, and that is what SDG Hub is built for.

In [ ]:
ex = json.loads((WS / "data/edgecopilot/train.jsonl").open().readline())["messages"]
print("NOTE  :", ex[1]["content"])
print("REPORT:", json.dumps(json.loads(ex[2]["content"]), indent=1))

## 4 · The rules, as code

Two small checkers. Every later notebook uses them, and the burst checker is also the **reward** for GRPO.
Each cell saves its code to the project volume, so training pods can import it too.

### Report checker: is the JSON report exactly right, and did the model invent a field?

In [ ]:
%%writefile /opt/app-root/src/workspace/code/report_check.py
"""Check a BD-SCR JSON report against the correct one."""
import json, re

OPTIONAL = ["mode", "position", "speed_kts", "heading_deg", "battery_pct", "visibility"]

def parse(text):
    """First JSON object in a model answer, or None."""
    text = re.sub(r"^```(?:json)?|```$", "", (text or "").strip(), flags=re.M)
    start, depth = text.find("{"), 0
    if start < 0:
        return None
    for i, ch in enumerate(text[start:], start):
        depth += (ch == "{") - (ch == "}")
        if depth == 0:
            try: return json.loads(text[start:i + 1])
            except json.JSONDecodeError: return None
    return None

def _canon(r):   # contacts and faults may come in any order
    key = lambda x: json.dumps(x, sort_keys=True, ensure_ascii=False)
    return {**r, **{k: sorted(r[k], key=key) for k in ("contacts", "faults") if isinstance(r.get(k), list)}}

def check(text, gold):
    p = parse(text)
    if not isinstance(p, dict):
        return {"valid": False, "exact": False, "invented": []}
    return {"valid": True, "exact": _canon(p) == _canon(gold),
            "invented": [k for k in OPTIONAL if gold.get(k) is None and p.get(k) is not None]}

def scorecard(answers, golds):
    res = [check(a, g) for a, g in zip(answers, golds)]
    pct = lambda xs: round(100 * sum(xs) / len(res), 1)
    return {"notes": len(res), "Valid JSON %": pct(r["valid"] for r in res),
            "Exact report %": pct(r["exact"] for r in res),
            "Invented a field %": pct(bool(r["invented"]) for r in res)}

### Burst-line checker: rules B1–B7 of the vessel manual

The burst line is one message of at most 160 characters. Many wordings are valid, so there is no single right
answer to copy, but every answer can be **checked**, rule by rule.

In [ ]:
%%writefile /opt/app-root/src/workspace/code/burst_rules.py
"""Rules B1-B7 for the satellite burst line, as code. Input `task` = {"priority": ..., "report": {...}}."""
import json, re

LIMIT = 160
MANUAL_CODES = {"E-102", "E-104", "E-117", "E-203", "E-221", "E-238", "E-246", "E-305", "E-312", "E-331",
                "E-408", "E-415", "E-502", "E-519"}

def required(task):
    """What the line must carry, as (rule, label, pattern)."""
    r, pri = task["report"], task["priority"]
    checks = [("B1", "priority", rf"^\s*{pri}\b"), ("B1", "vessel", rf"\b{re.escape(r['vessel'])}\b"),
              ("B2", "time", rf"\b{r['dtg'].split()[0]}\b")]
    for c in r.get("contacts") or []:
        if c["range_nm"] <= 3:                                   # B3: contacts within 3 nm, bearing + range
            rng = f"{c['range_nm']:g}"
            checks.append(("B3", f"contact {c['bearing_deg']:03d}",
                           rf"\b{c['bearing_deg']:03d}\b[^/]{{0,12}}?\b{re.escape(rng)} ?NM\b"))
            if c.get("ais") is not True:
                tag = "OFF" if c.get("ais") is False else "UNK"
                checks.append(("B3", f"AIS {tag}", rf"\bAIS {tag}\b"))
    for f in r.get("faults") or []:                              # B4: fault codes, unknown ones marked UNK
        unk = f["code"] not in MANUAL_CODES
        checks.append(("B4", f["code"] + (" UNK" if unk else ""),
                       rf"\b{re.escape(f['code'])}\b" + (r" ?UNK\b" if unk else r"(?! ?UNK\b)")))
    if r.get("battery_pct") is not None and r["battery_pct"] < 30:   # B5: low battery
        checks.append(("B5", "battery", rf"\bBATT ?{r['battery_pct']}\b"))
    return checks

def _allowed_numbers(r):                                         # B6: no number that is not in the report
    nums = set(re.findall(r"\d+(?:\.\d+)?", json.dumps({k: v for k, v in r.items() if k != "position"})))
    nums |= {f"{c['bearing_deg']:03d}" for c in r.get("contacts") or []} | {r["dtg"].split()[0][:-1], "3", "30"}
    return {float(n) for n in nums}

def check(text, task):
    """Rule-by-rule result for one answer: list of (rule, label, passed)."""
    line = (text or "").strip().splitlines()[-1].strip() if (text or "").strip() else ""
    out = [(rule, label, bool(re.search(rx, line))) for rule, label, rx in required(task)]
    stray = [n for n in re.findall(r"(?<![A-Z])\d+(?:\.\d+)?", line.replace("S7-", "S7 "))
             if float(n) not in _allowed_numbers(task["report"])]
    out.append(("B6", "no invented number", not stray))
    out += [("B7", "capitals", line == line.upper() and bool(line)),
            ("B7", f"<= {LIMIT} characters", 0 < len(line) <= LIMIT),
            ("B7", "ends with OUT", line.rstrip(". ").endswith("OUT"))]
    return out

def reward(text, task):
    """0..1: half for the share of rules met, half for meeting all of them; halved if too long."""
    if not (text or "").strip(): return 0.0
    res = check(text, task)
    score = 0.5 * sum(ok for *_, ok in res) / len(res) + 0.5 * all(ok for *_, ok in res)
    too_long = not next(ok for _, label, ok in res if label.startswith("<="))
    return round(score * (0.5 if too_long else 1.0), 4)

def reward_fn(data_source, solution_str, ground_truth):
    """The signature Training Hub's lora_grpo calls."""
    return reward(solution_str, json.loads(ground_truth))

def scorecard(answers, tasks):
    res = [check(a, t) for a, t in zip(answers, tasks)]
    pct = lambda xs: round(100 * sum(xs) / len(res), 1)
    return {"reports": len(res), "All rules met %": pct(all(ok for *_, ok in r) for r in res),
            "Missed a required item %": pct(any(not ok for rule, _, ok in r if rule <= "B5") for r in res),
            "Invented a number %": pct(not ok for r in res for rule, _, ok in r if rule == "B6"),
            "Mean reward": round(sum(reward(a, t) for a, t in zip(answers, tasks)) / len(res), 3)}

In [ ]:
import sys; sys.path.insert(0, str(WS / "code"))
import burst_rules
task = json.loads((WS / "data/burst/test.jsonl").open().readline())["ground_truth"]
task = json.loads(task)
print("PRIORITY:", task["priority"]); print("REPORT  :", json.dumps(task["report"])[:300], "...\n")
for line in ["FLASH S7-03 OUT", "this is not a burst line"]:
    print(repr(line), "-> reward", burst_rules.reward(line, task))
    for rule, label, ok in burst_rules.check(line, task):
        print(f"   {'PASS' if ok else 'FAIL'}  {rule}  {label}")

## 5 · SDG Hub: a teacher writes burst lines, the checker keeps the good ones

An SDG Hub **flow** is a YAML file of blocks that run in order. Our flow has three:

`PromptBuilderBlock` (fill the prompt) → `LLMChatBlock` (call the teacher model) → `LLMResponseExtractorBlock` (take the answer)

Built-in flows also add judge and filter blocks. Here the filter is our rule checker, applied right after the flow.

In [ ]:
%pip install -q sdg-hub      # served from Red Hat's vetted package index in OpenShift AI workbenches

In [ ]:
from sdg_hub import Flow, FlowRegistry
FlowRegistry.discover_flows()                   # built-in flows you can start from
print((REPO / "sdg_flows/burst_teacher/flow.yaml").read_text())

### The teacher model

Any OpenAI-compatible endpoint works. Best practice is a larger model **served in this project**, so no data leaves
it: for example Granite 4.1-8B, deployed the same way notebook 04 deploys the 3B model. We used Granite 4.1-8B.

No teacher yet? Set `TEACHER_URL = ""` and the next cells use the teacher answers we generated (`data/burst/`).

In [ ]:
TEACHER_URL   = ""                                # e.g. "https://granite-41-8b-kserve-workload-svc.<project>.svc:8000/v1"
TEACHER_MODEL = "granite-4.1-8b"
N_REPORTS     = 400                               # all training reports; use 40 for a quick test

In [ ]:
import pandas as pd
rows = [json.loads(l) for l in (WS / "data/burst/train.jsonl").open()][:N_REPORTS]
seed = pd.DataFrame([{"rules": r["messages"][0]["content"], "task": r["messages"][1]["content"]} for r in rows])

if TEACHER_URL:
    import litellm; litellm.ssl_verify = False     # in-cluster services use the cluster's own certificate
    flow = Flow.from_yaml(str(REPO / "sdg_flows/burst_teacher/flow.yaml"))
    flow.set_model_config(model=f"hosted_vllm/{TEACHER_MODEL}", api_base=TEACHER_URL, api_key="none")
    t0 = time.time()
    out = flow.generate(seed, max_concurrency=16)
    answers = [str(a).strip() for a in out["burst_content"]]
    print(f"teacher wrote {len(answers)} burst lines in {time.time() - t0:.0f} s")
else:
    teacher = [json.loads(l) for l in (WS / "data/burst/train-sft-teacher.jsonl").open()][:N_REPORTS]
    answers = [t["messages"][-1]["content"] for t in teacher]
    print(f"using {len(answers)} teacher answers from the repository")

In [ ]:
# The filter: keep only answers that meet every rule
tasks = [json.loads(r["ground_truth"]) for r in rows]
keep = [all(ok for *_, ok in burst_rules.check(a, t)) for a, t in zip(answers, tasks)]
print(burst_rules.scorecard(answers, tasks))
print(f"kept {sum(keep)} of {len(keep)}")

with open(WS / "data/burst/sft-teacher.jsonl", "w") as all_f, open(WS / "data/burst/sft-filtered.jsonl", "w") as ok_f:
    for r, a, k in zip(rows, answers, keep):
        line = json.dumps({"messages": r["messages"][:2] + [{"role": "assistant", "content": a}]}, ensure_ascii=False) + "\n"
        all_f.write(line)
        if k: ok_f.write(line)

bad = next(i for i, k in enumerate(keep) if not k)
print("\nA rejected answer:", answers[bad])
print([f"{rule} {label}" for rule, label, ok in burst_rules.check(answers[bad], tasks[bad]) if not ok])

In [ ]:
# Record the dataset in MLflow, next to the training runs that will use it
import mlflow
with mlflow.start_run(run_name="sdg-burst-teacher"):
    mlflow.log_params({"teacher": TEACHER_MODEL if TEACHER_URL else "repository file", "reports": len(rows)})
    mlflow.log_metrics({"teacher_answers": len(answers), "kept_after_checker": sum(keep)})
    mlflow.log_artifact(str(WS / "data/burst/sft-filtered.jsonl"))

## What we measured

| | Result |
|---|---|
| Teacher (Granite 4.1-8B) answers meeting every rule | 173 of 400 (43%), mostly missed contacts |
| SFT on **all** teacher answers | 31.5% of test messages meet every rule (worse than untuned, 42.5%) |
| SFT on the **173 checked** answers | 88.0% |

Filtering the teacher's output is what makes it useful. **Next:** notebook 02 picks the training method.